<a href="https://colab.research.google.com/github/broadinstitute/BE3D/blob/main/examples/BE3Dv9_MultiScreen_MEN1_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# README
https://github.com/broadinstitute/BE3D

# Setup

In [ ]:
# @title Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')
from google.colab import output
output.enable_custom_widget_manager()


In [ ]:
# @title Install DSSP and ClustalO

! apt-get update
! apt-get install dssp clustalo


In [ ]:
# @title Install MUSCLE

! wget https://github.com/rcedgar/muscle/releases/download/v5.3/muscle-linux-x86.v5.3
! chmod +x muscle-linux-x86.v5.3
! mv muscle-linux-x86.v5.3 muscle


In [ ]:
# @title Install BE3D

! pip install git+https://github.com/broadinstitute/beclust3d-public.git
print('beclust3d installed at:')
! pip show beclust3d


In [ ]:
# @title Import packages

import os
import sys
import yaml
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import gc
import uuid
import ipywidgets as widgets
from IPython.display import display

from IPython.display import Image, display, SVG
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from google.colab import files
import shutil

from beclust3d import *

import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

plt.rcParams['font.family'] = 'DejaVu Sans'


In [ ]:
# @title Download relevant files for running BE3D

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_display_yaml.py -O be3d_display_yaml.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_helper.py -O be3d_helper.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/be3d_qa.py -O be3d_qa.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_calculate_lfc3d.py -O beclust3d_calculate_lfc3d.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_nonaggregate.py -O beclust3d_nonaggregate.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/beclust3d_characterization.py -O beclust3d_characterization.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/bemetaclust3d_metaaggregate.py -O bemetaclust3d_metaaggregate.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/bemetaclust3d_characterization.py -O bemetaclust3d_characterization.py

! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/be3d_config_editor.py -O be3d_config_editor.py
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/feature/optimize-colab-notebooks/examples/be3d_plotly.py -O be3d_plotly.py


# BE3D Inputs (MEN1)

In [ ]:
# @title Download relevant files for MEN1

! mkdir MEN1/
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/yaml/MEN1-Perner2023.yaml -O MEN1/MEN1-Perner2023.yaml
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/men1.fasta -O MEN1/men1.fasta
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/pdb/men1.pdb -O MEN1/men1.pdb
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/PernerNature2023-MOLM13-Screen.tsv -O MEN1/PernerNature2023-MOLM13-Screen.tsv
! wget https://raw.githubusercontent.com/broadinstitute/BE3D/refs/heads/main/examples/data/PernerNature2023-MV411-Screen.tsv -O MEN1/PernerNature2023-MV411-Screen.tsv


In [ ]:
# @title Edit BE3D Inputs (interactive)
from be3d_config_editor import edit_yaml_config

# @markdown Path to downloaded or user-input .yaml configuration file
yaml_config_filepath = "MEN1/MEN1-Perner2023.yaml" # @param {type:"string"}

# @markdown Fill in / adjust the fields below (add or remove screen files in the
# @markdown **Screens** section), then click **Save config** before moving on.
editor = edit_yaml_config(yaml_config_filepath)
editor


In [ ]:
# @title Load Saved Config
# @markdown Run this cell **after** clicking "Save config" above.
with open(yaml_config_filepath, 'r') as file:
    input_dict = yaml.safe_load(file)

output_dir = input_dict['output_dir']
input_gene = input_dict['input_gene']
print(f"Loaded config for {input_gene}. Output directory: {output_dir}")


# BE3D Run (MEN1)

In [ ]:
# @title BE-QA and Visualizations
# @markdown Description

! python be3d_qa.py {yaml_config_filepath}

from be3d_plotly import plot_hypothesis_qa, plot_violin_by_muttype, show_side_by_side

hyp_fig = plot_hypothesis_qa(output_dir, pthr=input_dict.get('pthr', {}).get('single_screen', 0.05))

screens = input_dict['screens']
if isinstance(screens, str):
    screens = [s.strip() for s in screens.split(',')]
database = input_dict.get('database', {})
violin_figs = [
    plot_violin_by_muttype(
        input_dict['screen_dir'], screen_file,
        mut_col=database.get('mut_col', 'Mut_type'),
        val_col=database.get('val_col', 'sgRNA_score'),
        gene_col=database.get('gene_col'), gene_name=input_gene,
    )
    for screen_file in screens
]
violin_figs = [f for f in violin_figs if f is not None]

# Hypothesis QA scatter side by side with the first violin plot; any
# additional per-screen violin plots (multi-screen configs) shown below.
show_side_by_side(hyp_fig, violin_figs[0] if violin_figs else None)
for fig in violin_figs[1:]:
    fig.show()


## BE-Clust3D and Visualizations

In [ ]:
# @title BE-Clust3D and Visualizations
# @markdown Description

! python beclust3d_calculate_lfc3d.py {yaml_config_filepath}
! python beclust3d_nonaggregate.py {yaml_config_filepath}
from beclust3d.helpers.visualization.g2p import g2p_formatted_hit_cluster

screens = input_dict['screens']
if isinstance(screens, str):
    screens = [s.strip() for s in screens.split(',')]
screen_names = [s.split('.')[0] for s in screens]
gene_list = [input_gene] * len(screen_names)

pthr = input_dict.get('pthr', {})
single_pthr = str(pthr.get('single_screen', 0.05)).split('.')[1]
multi_pthr  = str(pthr.get('multi_screen', 0.001)).split('.')[1]
conservation_run = input_dict.get('conservation', {}).get('run', False)

g2p_formatted_hit_cluster(
    output_dir, gene_list, screen_names,
    lfc_pthr=single_pthr, lfc3d_pthr=single_pthr, meta_pthr=multi_pthr,
    function_for_meta=False,
    conservation=conservation_run,
    input_gene=input_gene,
)


In [ ]:
# @title Visualization (select a screen)
# @markdown Choose a screen from the dropdown to view its LFC, LFC3D, and cluster plots.

from ipywidgets import interact, Dropdown
from be3d_plotly import plot_score_scatter, plot_cluster_3d, show_side_by_side

def show_lfc3d(screen_name):
    lfc_fig = plot_score_scatter(output_dir, input_gene, screen_name, score_type='LFC', pthr_str=single_pthr)
    lfc3d_fig = plot_score_scatter(output_dir, input_gene, screen_name, score_type='LFC3D', pthr_str=single_pthr)
    show_side_by_side(lfc_fig, lfc3d_fig, width=1000)

    cluster_fig = plot_cluster_3d(output_dir, input_gene, screen_name, score_type='LFC3D',
                                   direction='Positive', pthr_str=multi_pthr, dist='6A')
    if cluster_fig is not None:
        cluster_fig.show()

interact(show_lfc3d, screen_name=Dropdown(options=screen_names, description='Screen:'));


## BEClust3D Characterization and Visualizations

In [ ]:
# @title BEClust3D Characterization
! python beclust3d_characterization.py {yaml_config_filepath}


In [ ]:
# @title Visualization (select a screen)
# @markdown Choose a screen from the dropdown to view its characterization plots.

from ipywidgets import interact, Dropdown
from be3d_plotly import (
    plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    show_side_by_side,
)

def show_characterization(screen_name):
    lfc_lfc3d_fig = plot_lfc_lfc3d_scatter(output_dir, input_gene, screen_name, pthr_str='05')
    plddt_rsa_fig = plot_plddt_rsa_scatter(output_dir, input_gene, screen_name)
    show_side_by_side(lfc_lfc3d_fig, plddt_rsa_fig, width=1000)

    for fig in [
        plot_domain_barplot(output_dir, input_gene, input_dict['input_uniprot'], screen_name, pthr_str='05'),
        plot_plddt_dis_barplot(output_dir, input_gene, screen_name, pthr_str='05'),
        plot_enrichment_test(output_dir, input_gene, screen_name, score_type='LFC3D', pthr_str='05'),
    ]:
        if fig is not None:
            fig.show()

interact(show_characterization, screen_name=Dropdown(options=screen_names, description='Screen:'));


## BE-MetaClust3D and Visualizations

In [ ]:
# @markdown Description

! python bemetaclust3d_metaaggregate.py {yaml_config_filepath}

# display(SVG(filename=f'{output_dir}meta-aggregate/plots/{input_gene}_LFC_cutoff05_scatter_cutoff.svg'))
display(SVG(filename=f'{output_dir}meta-aggregate/plots/{input_gene}_LFC3D_cutoff05_scatter_cutoff.svg'))
display(SVG(filename=f'{output_dir}cluster_LFC3D/plots/{input_gene}_Meta_LFC3D_Positive_Dendrogram_p<0.001_6A.svg'))

## BE-MetaClust3D Characterization and Visualizations

In [ ]:
! python bemetaclust3d_characterization.py {yaml_config_filepath}

display(Image(filename=f'{output_dir}characterization/plots/{input_gene}_LFC_LFC3D_scatter_05_Meta.png'))
display(Image(filename=f'{output_dir}characterization/plots/{input_gene}_pLDDT_RSA_scatter.png'))
display(Image(filename=f'{output_dir}characterization/plots/{input_gene}_Count_pLDDT_dis_barplot_LFC3D_05.png'))
display(Image(filename=f'{output_dir}characterization/plots/{input_gene}_enrichment_test_LFC3D_05.png'))

# G2P Visualizations

The steps below map your BE3D results onto an interactive 3D protein structure using the [Genomics 2 Proteins (G2P) portal](https://g2p.broadinstitute.org). You will download the BE3D output files, upload them to the portal, and explore base-editing scores and clusters directly on the structure.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, FileLink
import os
import shutil
from google.colab import files

def download_files_button(path):
    output_filename = f"{os.path.basename(path)}_files.zip"
    shutil.make_archive(output_filename.replace('.zip', ''), 'zip', path)

    print(f"Created {output_filename}. Click the button to download.")

    button = widgets.Button(description=f"Download {output_filename}")
    output = widgets.Output()

    def on_button_clicked(b):
        with output:
            files.download(output_filename)

    button.on_click(on_button_clicked)
    display(button, output)

In [ ]:
# @title 1. Download the G2P visualization files
# @markdown Run this cell, then click the button to download the BE3D output files prepared for G2P.
download_files_button('/content/MEN1/g2p_visualization')

In [ ]:
# @title 2. Open the G2P portal interactive module
# @markdown 1. Go to https://g2p.broadinstitute.org
# @markdown 2. Click **Interactive Module**

from IPython.display import Image
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/2_g2p.png", width=600)

In [ ]:
# @title 3. Start the structure-mapping workflow
# @markdown Click **Start with a gene/protein identifier** to map BE3D results onto a 3D structure using G2P.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/3_select_first_interactive_module.png", width=600)

In [ ]:
# @title 4. Select a gene
# @markdown Enter a gene name and click **Proceed**.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/4_select_a_gene.png", width=600)

In [ ]:
# @title 5. Select a protein structure
# @markdown Choose a 3D protein structure from the PDB, AlphaFold, or your own uploaded structure.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/5_select_a_pdb.png", width=600)

In [ ]:
# @title 6. Upload your BE3D results (.tsv)
# @markdown Click **Upload your data** and select the BE3D results file downloaded in step 1.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/6_upload_BE3D_tsv.png", width=600)

In [ ]:
# @title 7. Filter the columns of interest
# @markdown 1. Click **Filter Columns** and keep the columns you want to visualize.
# @markdown 2. Set the data type to **features** for any `... hit cluster` column.

Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/7_filter_columns.png", width=600)

In [ ]:
# @title 8. Explore the visualization
# @markdown Your BE3D scores and clusters are now mapped onto the 3D structure — rotate, zoom, and explore.
Image(url="https://raw.githubusercontent.com/broadinstitute/BE3D/main/examples/imgs/8_done.png", width=600)